# HUNTLITE Threat Detection Pipeline

This notebook documents the machine learning pipeline developed for HUNTLITE, a lightweight SOC-style detection and analysis tool designed to guide beginner users through threat triage, investigation, and incident reporting.

The goal of this phase was to prepare a cybersecurity log dataset, build a Keras-based classification model, and export the trained artifacts for later integration into the Streamlit application and AI Coach workflow.

## Dataset Source

This project uses the **Cybersecurity Threat Detection Logs Dataset** from Kaggle.

Author: Aryan208  
Source: https://www.kaggle.com/datasets/aryan208/cybersecurity-threat-detection-logs

### Dataset Description
The dataset contains labeled cybersecurity log data including:
- Network protocol
- Connection behavior
- Log source
- Request paths
- Threat labels

### Use in this Project
The dataset is cleaned, encoded, and used to train a neural network model using **Keras** to support automated threat detection in the HUNT-LITE security analysis platform.

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
import os

PROJECT_ROOT = "/content/drive/MyDrive/Huntlite_project"
RAW_DATA_PATH = f"{PROJECT_ROOT}/data/raw/cybersecurity_threat_detection_logs.csv"
PROCESSED_DIR = f"{PROJECT_ROOT}/data/processed"
MODELS_DIR = f"{PROJECT_ROOT}/models"
ARTIFACTS_DIR = f"{PROJECT_ROOT}/artifacts"

os.makedirs(PROCESSED_DIR, exist_ok=True)
os.makedirs(MODELS_DIR, exist_ok=True)
os.makedirs(ARTIFACTS_DIR, exist_ok=True)

In [3]:
!pip -q install pandas numpy scikit-learn joblib

In [4]:
import json
import joblib
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder, LabelEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import classification_report, confusion_matrix

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

## Data Loading and Initial Inspection

The original cybersecurity threat detection logs dataset was loaded from Google Drive into Google Colab for large-scale processing.

Initial inspection was performed to understand:
- the total number of records and features
- column data types
- missing values
- duplicate records
- class distribution in the target label

This helped confirm that the dataset was large, structured, and suitable for machine learning, while also revealing a strong class imbalance between benign, suspicious, and malicious events.

In [5]:
df = pd.read_csv(RAW_DATA_PATH)
print(df.shape)
df.head()

(6000000, 10)


,timestamp,source_ip,dest_ip,protocol,action,threat_label,log_type,bytes_transferred,user_agent,request_path
0,2024-05-01T00:00:00,192.168.1.125,192.168.1.124,TCP,blocked,benign,firewall,10889,Nmap Scripting Engine,/
1,2024-07-18T00:00:00,192.168.1.201,192.168.1.201,ICMP,blocked,benign,application,36522,Nmap Scripting Engine,/
2,2024-04-07T00:00:00,192.168.1.248,192.168.1.15,HTTP,allowed,benign,application,20652,Mozilla/5.0 (Windows NT 10.0; Win64; x64) Appl...,/login
3,2024-10-26T00:00:00,192.168.1.236,192.168.1.219,HTTP,allowed,benign,application,5350,Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7...,/login
4,2024-10-31T00:00:00,192.168.1.221,192.168.1.61,ICMP,allowed,benign,application,40691,Mozilla/5.0 (Windows NT 10.0; Win64; x64) Appl...,/


In [6]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 6000000 entries, 0 to 5999999
Data columns (total 10 columns):
 #   Column             Dtype 
---  ------             ----- 
 0   timestamp          object
 1   source_ip          object
 2   dest_ip            object
 3   protocol           object
 4   action             object
 5   threat_label       object
 6   log_type           object
 7   bytes_transferred  int64 
 8   user_agent         object
 9   request_path       object
dtypes: int64(1), object(9)
memory usage: 457.8+ MB


In [7]:
df.isnull().sum().sort_values(ascending=False).head(20)

,0
timestamp,0
source_ip,0
dest_ip,0
protocol,0
action,0
threat_label,0
log_type,0
bytes_transferred,0
user_agent,0
request_path,0


In [8]:
df.duplicated().sum()

np.int64(0)

In [9]:
df.columns.tolist()

['timestamp',
 'source_ip',
 'dest_ip',
 'protocol',
 'action',
 'threat_label',
 'log_type',
 'bytes_transferred',
 'user_agent',
 'request_path']

In [10]:
df.describe(include="all").T

,count,unique,top,freq,mean,std,min,25%,50%,75%,max
timestamp,6000000,365,2024-06-17T00:00:00,16777,NaN,NaN,NaN,NaN,NaN,NaN,NaN
source_ip,6000000,354,59.211.9.207,18295,NaN,NaN,NaN,NaN,NaN,NaN,NaN
dest_ip,6000000,254,192.168.1.5,24140,NaN,NaN,NaN,NaN,NaN,NaN,NaN
protocol,6000000,7,TCP,1497493,NaN,NaN,NaN,NaN,NaN,NaN,NaN
action,6000000,2,allowed,3000646,NaN,NaN,NaN,NaN,NaN,NaN,NaN
threat_label,6000000,3,benign,5517611,NaN,NaN,NaN,NaN,NaN,NaN,NaN
log_type,6000000,3,application,2001768,NaN,NaN,NaN,NaN,NaN,NaN,NaN
bytes_transferred,6000000.0,NaN,NaN,NaN,25046.492221,14401.631288,100.0,12584.0,25036.0,37519.0,50000.0
user_agent,6000000,5,curl/7.64.1,1200739,NaN,NaN,NaN,NaN,NaN,NaN,NaN
request_path,6000000,228,/,2741075,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [11]:
target_col = "threat_label"
df[target_col].value_counts(dropna=False)

,count
threat_label,
benign,5517611
suspicious,360883
malicious,121506


In [12]:
df[target_col].value_counts(normalize=True, dropna=False) * 100

,proportion
threat_label,
benign,91.960183
suspicious,6.014717
malicious,2.025100


## Feature Selection and Removal of Unnecessary Columns

Not all columns in the dataset were useful for model training.

Columns such as timestamp, source IP, and destination IP were removed from the machine learning dataset because they primarily act as identifiers rather than stable behavioral indicators. While these values are useful for investigation and reporting in the user interface, they are less useful for generalizable machine learning.

The remaining features were selected because they better represent network and application behavior:
- protocol
- action
- log type
- bytes transferred
- user agent
- request path

In [13]:
drop_cols = [
    "timestamp",
    "source_ip",
    "dest_ip"
]

In [14]:
df_ml = df.drop(columns=drop_cols)

In [15]:
df_ml.head()

,protocol,action,threat_label,log_type,bytes_transferred,user_agent,request_path
0,TCP,blocked,benign,firewall,10889,Nmap Scripting Engine,/
1,ICMP,blocked,benign,application,36522,Nmap Scripting Engine,/
2,HTTP,allowed,benign,application,20652,Mozilla/5.0 (Windows NT 10.0; Win64; x64) Appl...,/login
3,HTTP,allowed,benign,application,5350,Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7...,/login
4,ICMP,allowed,benign,application,40691,Mozilla/5.0 (Windows NT 10.0; Win64; x64) Appl...,/


In [16]:
df_ml.to_csv(f"{PROCESSED_DIR}/huntlite_ml_dataset.csv", index=False)

In [17]:
target_col = "threat_label"

X = df_ml.drop(columns=[target_col])
y = df_ml[target_col]

print("Feature shape:", X.shape)
print("Target shape:", y.shape)

Feature shape: (6000000, 6)
Target shape: (6000000,)


## Class Imbalance Analysis

The target label showed a strong imbalance across classes, with benign traffic representing the majority of the dataset and malicious traffic representing the smallest class.

This imbalance reflects real-world cybersecurity environments, where normal activity heavily outweighs attack events. However, such imbalance can bias a model toward predicting the majority class too often.

To address this issue without discarding data, class weighting was used during model training so that underrepresented classes contributed more strongly to the learning process.

## Encoding and Preprocessing

The dataset contained both categorical and numerical features.

Categorical features were encoded using one-hot encoding so that they could be used by the neural network. The numerical feature bytes_transferred was scaled to improve training stability and keep feature values on a comparable range.

A preprocessing pipeline was created and saved so that the same exact transformations used during training can also be reused later inside the HUNTLITE Streamlit application for inference.

In [18]:
y.value_counts()

,count
threat_label,
benign,5517611
suspicious,360883
malicious,121506


In [19]:
from sklearn.preprocessing import LabelEncoder

label_encoder = LabelEncoder()

y_encoded = label_encoder.fit_transform(y)

print(label_encoder.classes_)

['benign' 'malicious' 'suspicious']


In [20]:
joblib.dump(label_encoder, f"{ARTIFACTS_DIR}/label_encoder.joblib")

['/content/drive/MyDrive/Huntlite_project/artifacts/label_encoder.joblib']

In [21]:
numeric_cols = X.select_dtypes(include=["int64", "float64"]).columns.tolist()
categorical_cols = X.select_dtypes(include=["object"]).columns.tolist()

print("Numeric columns:", numeric_cols)
print("Categorical columns:", categorical_cols)

Numeric columns: ['bytes_transferred']
Categorical columns: ['protocol', 'action', 'log_type', 'user_agent', 'request_path']


## Train/Test Split

The processed dataset was divided into training and testing sets using a stratified split.

Stratification was used to preserve the original class proportions across both sets. This ensured that model evaluation would reflect the same class distribution observed in the original dataset.

In [22]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y_encoded,
    test_size=0.2,
    random_state=42,
    stratify=y_encoded
)

print("Training size:", X_train.shape)
print("Test size:", X_test.shape)

Training size: (4800000, 6)
Test size: (1200000, 6)


In [23]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler

preprocessor = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), numeric_cols),
        ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_cols)
    ]
)

In [24]:
X_train_processed = preprocessor.fit_transform(X_train)
X_test_processed = preprocessor.transform(X_test)

print("Processed train shape:", X_train_processed.shape)
print("Processed test shape:", X_test_processed.shape)

Processed train shape: (4800000, 246)
Processed test shape: (1200000, 246)


In [50]:
import joblib

joblib.dump(preprocessor, f"{ARTIFACTS_DIR}/preprocessor.joblib")
print("Preprocessor saved successfully.")

Preprocessor saved successfully.


## Class Weighting Strategy

Because the dataset was highly imbalanced, class weights were computed and applied during training.

This approach allows the model to pay greater attention to minority classes such as malicious and suspicious traffic while still learning from the full dataset. It is a more practical solution than randomly discarding large portions of benign data.

In [25]:
import numpy as np
from sklearn.utils.class_weight import compute_class_weight

classes = np.unique(y_train)

weights = compute_class_weight(
    class_weight="balanced",
    classes=classes,
    y=y_train
)

class_weight_dict = dict(zip(classes, weights))

print(class_weight_dict)

{np.int64(0): np.float64(0.36247569996889506), np.int64(1): np.float64(16.460058638958902), np.int64(2): np.float64(5.541970031797053)}


In [26]:
X_train_dense = X_train_processed.toarray()
X_test_dense = X_test_processed.toarray()

print(X_train_dense.shape)

(4800000, 246)


## Keras Model Design

A neural network was built using Keras to classify records into benign, suspicious, and malicious categories.

The architecture consisted of:
- an input layer based on the transformed feature size
- a dense hidden layer with ReLU activation
- dropout for regularization
- a second dense hidden layer
- a softmax output layer for multi-class classification

This design provided a lightweight but effective architecture suitable for integration into the HUNTLITE application.

In [30]:
input_dim = X_train_dense.shape[1]
num_classes = len(np.unique(y_train))

print("Input dimension:", input_dim)
print("Number of classes:", num_classes)

Input dimension: 246
Number of classes: 3


In [31]:
model = keras.Sequential([

    layers.Input(shape=(input_dim,)),

    layers.Dense(128, activation="relu"),
    layers.Dropout(0.3),

    layers.Dense(64, activation="relu"),
    layers.Dropout(0.2),

    layers.Dense(num_classes, activation="softmax")
])

In [32]:
model.compile(
    optimizer="adam",
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense (Dense)                   │ (None, 128)            │        31,616 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 64)             │         8,256 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 3)              │           195 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 40,067 (156.51 KB)

 Trainable params: 40,067 (156.51 KB)

 Non-trainable params: 0 (0.00 B)

## Model Training

The model was trained on the processed training dataset using the Adam optimizer and sparse categorical cross-entropy loss.

Early stopping was added to prevent unnecessary training once validation performance stopped improving. Class weights were also applied during training to reduce the impact of class imbalance.

## Model Evaluation

After training, the model was evaluated on the held-out test set using accuracy, classification metrics, and a confusion matrix.

The model achieved near-perfect performance on the dataset. Because this result is unusually strong, additional validation checks were performed to determine whether some features in the synthetic dataset were directly revealing the class label.

In [33]:
callbacks = [
    keras.callbacks.EarlyStopping(
        monitor="val_loss",
        patience=3,
        restore_best_weights=True
    )
]

history = model.fit(
    X_train_dense,
    y_train,
    validation_split=0.2,
    epochs=20,
    batch_size=1024,
    class_weight=class_weight_dict,
    callbacks=callbacks,
    verbose=1
)

Epoch 1/20
3750/3750 ━━━━━━━━━━━━━━━━━━━━ 16s 3ms/step - accuracy: 0.9728 - loss: 0.0981 - val_accuracy: 1.0000 - val_loss: 9.5376e-08
Epoch 2/20
3750/3750 ━━━━━━━━━━━━━━━━━━━━ 11s 3ms/step - accuracy: 1.0000 - loss: 1.4186e-05 - val_accuracy: 1.0000 - val_loss: 2.5344e-10
Epoch 3/20
3750/3750 ━━━━━━━━━━━━━━━━━━━━ 11s 3ms/step - accuracy: 1.0000 - loss: 1.3755e-06 - val_accuracy: 1.0000 - val_loss: 0.0000e+00
Epoch 4/20
3750/3750 ━━━━━━━━━━━━━━━━━━━━ 11s 3ms/step - accuracy: 1.0000 - loss: 3.3831e-06 - val_accuracy: 1.0000 - val_loss: 7.4506e-13
Epoch 5/20
3750/3750 ━━━━━━━━━━━━━━━━━━━━ 11s 3ms/step - accuracy: 1.0000 - loss: 1.3661e-07 - val_accuracy: 1.0000 - val_loss: 0.0000e+00
Epoch 6/20
3750/3750 ━━━━━━━━━━━━━━━━━━━━ 10s 3ms/step - accuracy: 1.0000 - loss: 1.2512e-07 - val_accuracy: 1.0000 - val_loss: 0.0000e+00


## Validation of Perfect Accuracy

To investigate the unusually high performance, feature-to-label relationships were examined using cross-tabulation.

This analysis showed that the request_path feature strongly and in some cases directly determined the threat label. For example, certain attack-like paths were always labeled malicious or suspicious, while the root path was always labeled benign.

This indicates that the dataset is synthetic and contains highly separable attack signatures. As a result, the model's perfect accuracy reflects the structure of the dataset rather than a claim of real-world production-grade detection.

In [34]:
test_loss, test_accuracy = model.evaluate(X_test_dense, y_test)

print("Test Loss:", test_loss)
print("Test Accuracy:", test_accuracy)

37500/37500 ━━━━━━━━━━━━━━━━━━━━ 65s 2ms/step - accuracy: 1.0000 - loss: 0.0000e+00
Test Loss: 0.0
Test Accuracy: 1.0


In [35]:
y_pred_probs = model.predict(X_test_dense)
y_pred = np.argmax(y_pred_probs, axis=1)

37500/37500 ━━━━━━━━━━━━━━━━━━━━ 44s 1ms/step


In [36]:
print(classification_report(y_test, y_pred, target_names=label_encoder.classes_))

              precision    recall  f1-score   support

      benign       1.00      1.00      1.00   1103522
   malicious       1.00      1.00      1.00     24301
  suspicious       1.00      1.00      1.00     72177

    accuracy                           1.00   1200000
   macro avg       1.00      1.00      1.00   1200000
weighted avg       1.00      1.00      1.00   1200000



In [37]:
print(confusion_matrix(y_test, y_pred))

[[1103522       0       0]
 [      0   24301       0]
 [      0       0   72177]]


## Saved Artifacts for Streamlit Integration

At the end of the notebook, all core artifacts needed for deployment were saved:

- trained Keras model
- preprocessing pipeline
- label encoder
- label mapping
- feature metadata
- sample records for Streamlit testing

These files will be moved into the VS Code project and connected to the HUNTLITE detection engine and AI Coach so that the application can perform prediction and guide users through SOC-style triage.

In [38]:
model.save(f"{MODELS_DIR}/huntlite_threat_detection_model.keras")

In [39]:
feature_info = {
    "numeric_cols": numeric_cols,
    "categorical_cols": categorical_cols,
    "target_col": target_col
}

with open(f"{ARTIFACTS_DIR}/feature_info.json", "w") as f:
    json.dump(feature_info, f, indent=2)

In [40]:
label_map = dict(enumerate(label_encoder.classes_))

with open(f"{ARTIFACTS_DIR}/label_mapping.json", "w") as f:
    json.dump(label_map, f, indent=2)

In [41]:
sample_rows = X_test.head(100).copy()
sample_rows[target_col] = label_encoder.inverse_transform(y_test[:100])
sample_rows.to_csv(f"{PROCESSED_DIR}/huntlite_streamlit_test_samples.csv", index=False)

## Relevance to HUNTLITE

This machine learning phase supports the broader HUNTLITE goal of helping beginner users move through a realistic threat triage workflow.

Rather than functioning only as a classifier, the model serves as one component of a larger educational analysis system. Its predictions will be used by the AI Coach to explain likely threat categories, suggest triage steps, support investigation flow, and contribute to final incident reporting inside the Streamlit interface.